In [1]:
import os
import boto3
from sagemaker import get_execution_role
import shutil
from pprint import pprint
import time
import pandas as pd

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/ec2-user/.config/sagemaker/config.yaml


#### Constants

In [2]:
# function name
str_function_name = 'gen-xii-payload-parsing-jq'

#### 1. Create Container

#### Create `Dockerfile`

In [3]:
%%writefile Dockerfile

FROM public.ecr.aws/lambda/python:3.8

# update pip
RUN pip install --upgrade pip

# install dependencies from project folder
COPY requirements.txt  .
RUN  pip3 install -r requirements.txt --target "${LAMBDA_TASK_ROOT}"

# copy functions.py
COPY functions.py ${LAMBDA_TASK_ROOT}

# copy api.py
COPY api.py ${LAMBDA_TASK_ROOT}

# copy parser
COPY cls_parser.pkl ${LAMBDA_TASK_ROOT}

# copy function code
COPY lambda_function.py ${LAMBDA_TASK_ROOT}

# Set the CMD to your handler (could also be done as a parameter override outside of the Dockerfile)
CMD ["lambda_function.lambda_handler"] 

Writing Dockerfile


#### Write `requirements.txt`

In [4]:
%%writefile requirements.txt

pyarrow==9.0.0
fsspec==2022.10.0
s3fs==2022.10.0

tqdm==4.64.1
numpy==1.23.4
pandas==1.2.4
boto3==1.24.59

Writing requirements.txt


#### Copy files

In [5]:
list_str_filename = [
    'functions.py',
    'api.py',
    'cls_parser.pkl',
]
for str_filename in list_str_filename:
    # logic
    if str_filename == 'cls_parser.pkl':
        str_source = f'../01_create_parser/output/{str_filename}'
    else:
        str_source = f'../01_create_parser/{str_filename}'
    str_destination = f'./{str_filename}'
    # copy
    shutil.copyfile(str_source, str_destination)

#### Write `lambda_function.py`

In [6]:
%%writefile lambda_function.py

import pandas as pd
import pickle
import json

# lambda handler
def lambda_handler(event, context):
    # get the input
    int_rows_to_parse = int(event['row'])
    print(f'Parsing rows: {int_rows_to_parse}')
    
    # constants
    str_project = '20240423-gen-xii-payload-parsing'
        
    # load requests
    print('Loading requests...')
    str_filename = f'df_rows_{int_rows_to_parse}.gzip'
    str_uri = f's3://{str_project}/03_step_function/rows/{str_filename}'
    df = pd.read_parquet(str_uri)
    print(f'There are {df.shape[0]} requests for this lambda function to parse')
    
    # load parser
    print('Loading parser...')
    str_filename = 'cls_parser.pkl'
    str_local_path = f'./{str_filename}'
    cls_parser = pickle.load(open(str_local_path, 'rb'))
    
    # parse (get income, LN, and TU)
    print('Parsing requests...')
    list_X_raw = []
    for a, str_request in enumerate(df['strRequest']):
        # get bigAccountId
        int_bigaccountid = df['bigAccountId'].iloc[a]
        # get dtmFunded
        dtm_funded = df['dtmFunded'].iloc[a]
    
        # convert string request to dict
        dict_json_request = json.loads(str_request)
        # get data
        cls_parser.get_data(dict_json_request)
        # parse data
        cls_parser.parse_data()
        # create X
        cls_parser.create_x()
        # get X_raw
        X_raw = cls_parser.dict_output['X_raw']
        
        # assign
        X_raw['bigAccountId'] = int_bigaccountid
        X_raw['dtmFunded'] = dtm_funded
        # get nrows
        int_nrows = X_raw.shape[0]
        if int_nrows == 1:
            X_raw['BITDEBTOR'] = 1
        else:
            X_raw['BITDEBTOR'] = [1,0]
        
        # append
        list_X_raw.append(X_raw)
    
    # concat
    print('Concatenating raw data...')
    X_raw = pd.concat(list_X_raw)
    
    # save memeory
    del list_X_raw
    
    # write to s3 as parquet
    print('Writing raw data to s3...')
    # set nonnumeric to string
    for col in X_raw.columns:
        # if not numeric
        if X_raw[col].dtype not in ['int64', 'float64']:
            # set as string
            X_raw[col] = X_raw[col].astype(str)
        else:
            pass
    # write to gzip
    str_filename = f'X_raw_{int_rows_to_parse}.gzip'
    str_uri = f's3://{str_project}/03_step_function/parsed/{str_filename}'
    X_raw.to_parquet(str_uri, compression='gzip')

Writing lambda_function.py


#### Build image and push to ECR

In [7]:
%%sh

# name the image
image=gen-xii-payload-parsing-jq

# build image
docker build -t ${image} .

# get region
region=$(aws configure get region)
region=${region:-us-west-2}

# get account
account=$(aws sts get-caller-identity --query Account --output text)

# get full name
fullname="${account}.dkr.ecr.${region}.amazonaws.com/${image}:latest"

# get login command and execute it
aws ecr get-login-password --region "${region}" | docker login --username AWS --password-stdin "${account}".dkr.ecr."${region}".amazonaws.com

# create repository in ECR
aws ecr create-repository --repository-name "${image}" --image-scanning-configuration scanOnPush=true --image-tag-mutability MUTABLE

# tag image
docker tag  ${image} ${fullname}

# push image to ECR   
docker push ${fullname}

Sending build context to Docker daemon  93.18kB
Step 1/9 : FROM public.ecr.aws/lambda/python:3.8
3.8: Pulling from lambda/python
7515bf1ca88c: Pulling fs layer
3b179787afe0: Pulling fs layer
dd60359c7fee: Pulling fs layer
b51476fb673f: Pulling fs layer
ac93c5f26606: Pulling fs layer
79f038440c32: Pulling fs layer
79f038440c32: Waiting
b51476fb673f: Waiting
ac93c5f26606: Waiting
3b179787afe0: Download complete
dd60359c7fee: Download complete
b51476fb673f: Verifying Checksum
b51476fb673f: Download complete
79f038440c32: Verifying Checksum
79f038440c32: Download complete
ac93c5f26606: Verifying Checksum
ac93c5f26606: Download complete
7515bf1ca88c: Verifying Checksum
7515bf1ca88c: Download complete
7515bf1ca88c: Pull complete
3b179787afe0: Pull complete
dd60359c7fee: Pull complete
b51476fb673f: Pull complete
ac93c5f26606: Pull complete
79f038440c32: Pull complete
Digest: sha256:c6f810e33459a0805bcb813ea7ee398dff5a45ddc0e4cff42e4e8a0123adf5dc
Status: Downloaded newer image for public.ecr.a

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 229.9/229.9 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 505.5/505.5 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.8/79.8 kB 17.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.8/60.8 kB 13.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 240.9/240.9 kB 35.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.3/129.3 kB 25.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.9/143.9 kB 22.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 83.4/83.4 kB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 308.8/308.8 kB 36.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 11.4 MB/s eta 0:00:00
Removing intermediate container f42c9a30b72c
 ---> 26e12a857278
Step 5/9 : COPY functions.py ${LAMBDA_TASK_ROOT}
 ---> 4af36028fff1
Step 6/9 : COPY api.py ${LAMBDA_TASK_ROOT}
 ---> c8de7b2d06bc
Step 7

WARNING! Your password will be stored unencrypted in /home/ec2-user/.docker/config.json.
Configure a credential helper to remove this warning. See
https://docs.docker.com/engine/reference/commandline/login/#credentials-store



Login Succeeded



An error occurred (RepositoryAlreadyExistsException) when calling the CreateRepository operation: The repository with name 'gen-xii-payload-parsing-jq' already exists in the registry with id '836690756591'


The push refers to repository [836690756591.dkr.ecr.us-west-2.amazonaws.com/gen-xii-payload-parsing-jq]
285bf85ec821: Preparing
99bd8895e2ce: Preparing
8e7f052477ef: Preparing
9d29d2f5b270: Preparing
5af5ba8b0086: Preparing
d61db6024624: Preparing
4ebb1d9a134e: Preparing
1d708e6cabd7: Preparing
4fa6da82f992: Preparing
e8e5c6e3f5d5: Preparing
291463b8759f: Preparing
dfe3b1682705: Preparing
ecafc03674ff: Preparing
d61db6024624: Waiting
4ebb1d9a134e: Waiting
1d708e6cabd7: Waiting
4fa6da82f992: Waiting
e8e5c6e3f5d5: Waiting
291463b8759f: Waiting
ecafc03674ff: Waiting
dfe3b1682705: Waiting
285bf85ec821: Pushed
9d29d2f5b270: Pushed
99bd8895e2ce: Pushed
8e7f052477ef: Pushed
d61db6024624: Pushed
4ebb1d9a134e: Pushed
e8e5c6e3f5d5: Pushed
291463b8759f: Pushed
dfe3b1682705: Pushed
1d708e6cabd7: Pushed
4fa6da82f992: Pushed
5af5ba8b0086: Pushed
ecafc03674ff: Pushed
latest: digest: sha256:63541ac3635976a17dca0c24fd9244fad2246108f5d214c6d13b575291b6a12e size: 3044


#### 2. Create lambda function from image

In [8]:
# initialize class
cls_client_lambda = boto3.client('lambda')

In [9]:
# get role
str_role = get_execution_role()
print(f'Role: {str_role}')

Role: arn:aws:iam::836690756591:role/risk-ops-role


In [10]:
# delete it if it exists
try:
    dict_response = cls_client_lambda.delete_function(
        FunctionName=str_function_name,
    )
    pprint(dict_response)
except:
    pass

{'ResponseMetadata': {'HTTPHeaders': {'connection': 'keep-alive',
                                      'content-type': 'application/json',
                                      'date': 'Fri, 28 Jun 2024 16:50:29 GMT',
                                      'x-amzn-requestid': '068d68f5-e99b-4a2e-8358-557b0aa3372e'},
                      'HTTPStatusCode': 204,
                      'RequestId': '068d68f5-e99b-4a2e-8358-557b0aa3372e',
                      'RetryAttempts': 0}}


In [11]:
# create function
str_image_uri = f'836690756591.dkr.ecr.us-west-2.amazonaws.com/{str_function_name}:latest' # this must match what we name the image above
dict_response = cls_client_lambda.create_function(
    FunctionName=str_function_name,
    Role=str_role,
    Code={
        'ImageUri': str_image_uri,
    },
    Timeout=900, # 15 minutes is maximum
    MemorySize=1000, # 1000 mb == 1 gb
    Publish=True,
    PackageType='Image',
    Architectures=[
        'x86_64',
    ],
    EphemeralStorage={
        'Size': 1000, # 1000 mb == 1 gb
    },
)
pprint(dict_response)
time.sleep(40)

{'Architectures': ['x86_64'],
 'CodeSha256': '63541ac3635976a17dca0c24fd9244fad2246108f5d214c6d13b575291b6a12e',
 'CodeSize': 0,
 'Description': '',
 'EphemeralStorage': {'Size': 1000},
 'FunctionArn': 'arn:aws:lambda:us-west-2:836690756591:function:gen-xii-payload-parsing-jq',
 'FunctionName': 'gen-xii-payload-parsing-jq',
 'LastModified': '2024-06-28T16:50:29.379+0000',
 'LoggingConfig': {'LogFormat': 'Text',
                   'LogGroup': '/aws/lambda/gen-xii-payload-parsing-jq'},
 'MemorySize': 1000,
 'PackageType': 'Image',
 'ResponseMetadata': {'HTTPHeaders': {'connection': 'keep-alive',
                                      'content-length': '1213',
                                      'content-type': 'application/json',
                                      'date': 'Fri, 28 Jun 2024 16:50:30 GMT',
                                      'x-amzn-requestid': '9d03a5f6-4848-4534-b4d4-956d00bfb223'},
                      'HTTPStatusCode': 201,
                      'RequestId': '9d

#### Clean - up

In [12]:
list_str_filename = list_str_filename + ['Dockerfile', 'lambda_function.py', 'requirements.txt']

for str_file in list_str_filename:
    os.remove(str_file)